In [1]:
from langchain_community.document_loaders import PyPDFLoader

document = PyPDFLoader(file_path="./documents/cricket-laws-2019.pdf")
documents = document.load()

Ignoring wrong pointing object 9 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 39 0 (offset 0)
Ignoring wrong pointing object 390 0 (offset 0)
Ignoring wrong pointing object 392 0 (offset 0)


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, separators=["\n\n", "\n \n", "\n", ". ", " ", ""])
chunks = text_splitter.split_documents(documents=documents)

In [3]:
from collections import defaultdict
tagging_dict = defaultdict(int)

for chunk in chunks:
    source = chunk.metadata.get("source")
    page = chunk.metadata.get("page")
    block = f"{source}:{page}"
    chunk.metadata["id"] = "{block}:{chunk_id}".format(block=block, chunk_id=tagging_dict[block])
    tagging_dict[block] += 1


In [10]:
from langchain_ollama import OllamaEmbeddings
embeddings = OllamaEmbeddings(model="nomic-embed-text:v1.5")

In [6]:
from langchain_chroma import Chroma

vector_db = Chroma(collection_name="cricket_meta", embedding_function=embeddings, persist_directory="./chroma_data/chroma_default_db")
vector_db.add_documents(documents=chunks, ids=[chunk.metadata.get("id") for chunk in chunks])

['./documents/cricket-laws-2019.pdf:0:0',
 './documents/cricket-laws-2019.pdf:1:0',
 './documents/cricket-laws-2019.pdf:1:1',
 './documents/cricket-laws-2019.pdf:1:2',
 './documents/cricket-laws-2019.pdf:1:3',
 './documents/cricket-laws-2019.pdf:1:4',
 './documents/cricket-laws-2019.pdf:2:0',
 './documents/cricket-laws-2019.pdf:2:1',
 './documents/cricket-laws-2019.pdf:2:2',
 './documents/cricket-laws-2019.pdf:3:0',
 './documents/cricket-laws-2019.pdf:3:1',
 './documents/cricket-laws-2019.pdf:4:0',
 './documents/cricket-laws-2019.pdf:4:1',
 './documents/cricket-laws-2019.pdf:4:2',
 './documents/cricket-laws-2019.pdf:5:0',
 './documents/cricket-laws-2019.pdf:5:1',
 './documents/cricket-laws-2019.pdf:5:2',
 './documents/cricket-laws-2019.pdf:5:3',
 './documents/cricket-laws-2019.pdf:6:0',
 './documents/cricket-laws-2019.pdf:6:1',
 './documents/cricket-laws-2019.pdf:6:2',
 './documents/cricket-laws-2019.pdf:6:3',
 './documents/cricket-laws-2019.pdf:7:0',
 './documents/cricket-laws-2019.pd

In [12]:
user_query = "How many balls in a over?"

In [17]:
results = vector_db.similarity_search_with_score(query=user_query, k=5)
context_var = ""
for each in results:
    doc, _score = each
    context_var += doc.page_content + "\n"

print(context_var)

. 17.2 Start of an over An over has started when the bowler starts his/her run-up or, if there is no run-up, starts his/her action for the first delivery of that over. 17.3 Validity of balls 17.3.1 A ball shall not count as one of the 6 balls of the over unless it is delivered, even though, as in Law 41.16 (Non-striker leaving his/her ground early) a batsman may be dismissed or some other incident occurs without the ball having been delivered. 17.3.2 A ball delivered by the bowler shall not count as one of the 6 balls of the over 17.3.2.1 if it is called dead, or is to be considered dead, before the striker has had an opportunity to play it.  See Law 20.6 (Dead ball; ball counting as one of over). 17.3.2.2 if it is called dead in the circumstances of Law 20.4.2.6.  Note also the special provisions of Law 20.4.2.5 (Umpire calling and signalling Dead ball). 17.3.2.3 if it is a No ball.  See Law 21 (No ball). 17.3.2.4 if it is a Wide.  See Law 22 (Wide ball)
.   21.17 No ball not to count

In [27]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.1:latest", temperature=0.4)
parser = StrOutputParser()
prompt = PromptTemplate(template="""
Answer the question based only on the following context.

<context>
{context_var}              
</context>

Answer the question based on the above context: {user_question}                       
""", input_variables=["context_var", "user_question"])

In [28]:
chain = prompt | llm | parser

In [29]:
response = chain.invoke(input={
    "context_var": context_var, "user_question": user_query
})

In [32]:
response

'6 balls.'

In [41]:
user_query = "What does innings mean and how many innings are there in a test match?"
results = vector_db.similarity_search_with_score(query=user_query, k=10)
context_var = ""
for each in results:
    doc, _score = each
    context_var += doc.page_content + "\n"

print(context_var)

If an innings ends so that a new innings is to be started during the last hour of the match, the interval starts with the end of the innings and is to end 10 minutes later. 12.8.1 If this interval is already in progress at the start of the last hour then, to determine the number of overs to be bowled in the new innings, calculations are to be made as set out in 12.7. 12.8.2 If the innings ends after the last hour has started, two calculations are to be made, as set out in 12.8.3 and 12.8.4.  The greater of the numbers yielded by these two calculations is to be the minimum number of overs to be bowled in the new innings. 12.8.3 Calculation based on overs remaining: - At the conclusion of the innings, the number of overs that remain to be bowled, of the minimum in the last hour, to be noted. - If this is not a whole number it is to be rounded up to the next whole number
. 13.1.2.2 in a two-innings match similar agreements shall apply  to the first innings of each side or to the second in

In [42]:
chain.invoke(input={
    "context_var": context_var, "user_question": user_query
})

'According to the context, an "innings" refers to a period of batting by a team in a cricket match. It ends when the team is "all out" or when the captain declares the innings closed.\n\nAs for the number of innings in a test match, the context does not explicitly state that there are only two innings in a test match. However, it does mention "a two-innings match" and "the first innings of each side or to the second innings of each side" in Law 13.1.2.2, which suggests that a standard test match typically consists of two innings per team.'